[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/module-3-cleaning-and-exploring/notebook-3.1-handling-missing-values.ipynb)


# Module 3 — Handling Missing Values (Colab walkthrough)

A hands-on companion to the lecture on **handling missing values**, run on a small, **synthetic** Barcelona-Bicing-style table of per-station snapshots.

## What this notebook is for

This notebook is a **preview** of what the lecture will show you. Run it top to bottom — or just read the printed outputs — to get a feel for the work before the lecture explains it: finding the different ways a value can be *missing* in a real table, then deciding whether to **drop** or **fill** each kind and checking the table before and after. You don't need to follow every line of code yet. The goal is to **picture what we'll be doing**, so when the lecture walks through the *why*, you already have a concrete example in your head. Every step prints a number you can check.

### How to run in Google Colab
1. `Runtime ▸ Run all`. The notebook **reads the dataset straight from this course's public GitHub repo** — nothing to upload, nothing to generate.
2. Everything below reads that CSV and prints a checkable result for each thing the lecture shows.

> ⚠️ **The data is synthetic.** Rows are fabricated to be *plausible* and to match the columns, types, quirks, and numbers used in the Module 3 lecture scripts. It is **not** real Bicing data — real Bicing snapshots are ~370 MB (too big for Colab) and no public trip-level Bicing data exists, which is why this small synthetic stand-in is hosted in the course repo and read directly above.


In [1]:
import numpy as np
import pandas as pd

# This course's practice data lives in its public GitHub repo. We read it straight
# from there, so the notebook runs the same in Google Colab or anywhere online —
# nothing to upload, nothing to generate.
DATA_URL = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-3-cleaning-and-exploring/data/"
            "barcelona-bicing-synthetic/")


## Handling Missing Values

*Prompt idea:* "Summarize missing values across all columns — count nulls, empty
strings, and `s/n` values; then drop rows with missing coordinates and verify
before/after."

We load the file with `keep_default_na=False` so every **form** of missingness stays
visible — a default load would silently turn blanks into `NaN` and hide the distinction
between an empty string and a true null.


In [2]:
snap = pd.read_csv(f"{DATA_URL}station-snapshots-sample.csv", keep_default_na=False)
print("rows:", len(snap))

# Missingness has more than one shape:
lat = pd.to_numeric(snap["latitude"], errors="coerce")
lon = pd.to_numeric(snap["longitude"], errors="coerce")
n_nan_coords   = int((lat.isna() | lon.isna()).sum())
n_empty_nearby = int((snap["nearbyStations"].str.strip() == "").sum())
n_sn           = int((snap["streetNumber"].str.strip().str.lower() == "s/n").sum())
print("NaN coordinates (rows)      :", n_nan_coords)
print("empty-string nearbyStations :", n_empty_nearby)
print("'s/n' sentinel streetNumber :", n_sn)


rows: 1248
NaN coordinates (rows)      : 17
empty-string nearbyStations : 22
's/n' sentinel streetNumber : 30


In [3]:
# Strategy: DROP rows with missing coordinates (spatial analysis needs valid lat/lon).
df = snap.copy()
df["latitude"]  = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

before = len(df)
clean  = df.dropna(subset=["latitude", "longitude"])
after  = len(clean)
remaining = int(clean[["latitude", "longitude"]].isna().sum().sum())
print("rows before:", before)
print("rows after :", after)
print("remaining nulls in latitude/longitude:", remaining)


rows before: 1248
rows after : 1231
remaining nulls in latitude/longitude: 0


In [4]:
# Strategy: FILL a label column, and FLAG every imputation (guardrail).
# streetNumber is a label, so 's/n' can become 'Unknown' without inventing a number.
filled = snap.copy()
filled["streetNumber_imputed"] = filled["streetNumber"].str.strip().str.lower() == "s/n"
filled["streetNumber"] = filled["streetNumber"].where(
    ~filled["streetNumber_imputed"], "Unknown")
print("streetNumber 's/n' values filled with 'Unknown':", int(filled["streetNumber_imputed"].sum()))
print("imputation flag column added:", "streetNumber_imputed" in filled.columns)
print("Guardrail: coordinates were DROPPED (not imputed); never impute a prediction target.")


streetNumber 's/n' values filled with 'Unknown': 30
imputation flag column added: True
Guardrail: coordinates were DROPPED (not imputed); never impute a prediction target.


## Done — handling missing values

The missing-values lecture has run on the synthetic snapshots: the several forms of
missingness counted side by side, the 1,248 → 1,231 coordinate **drop** with zero
residual nulls, and the **fill-and-flag** strategy for the `s/n` label. Re-run any time —
the notebook reads the same published data, so the numbers don't change. Then try the
matching exercise yourself.
